# Tutorial 04: DP Optimizers - From Manual to Production-Ready

**Level**: Intermediate to Advanced
**Duration**: 60-75 minutes
**Prerequisites**: Tutorial 01-03 (Gradient Clipping, Noise, Manual DP-SGD)

---

## Overview

In Tutorial 03, you implemented DP-SGD **manually** with clipping + noise. This is great for learning,
but for production training you want **optimizers** that handle everything automatically.

**What you'll learn today**:
1. **Recap**: Manual DP-SGD from Tutorial 03
2. **DP-SGD optimizer**: Production-ready with automatic noise + accounting
3. **DP-AdamW**: Adaptive learning rates + weight decay for better convergence
4. **DP-Adam-AC** (Extra): Adaptive clipping for optimal privacy-utility tradeoff

**Key Takeaway**: Opaque's optimizers make DP training as easy as standard PyTorch training!

---

## The Evolution

```python
# Tutorial 03: Manual (3 steps per batch)
grads = clipped_grad_fn(params, batch)
grads = add_gaussian_noise(grads, stddev)
params = manual_sgd_update(params, grads, lr)

# Tutorial 04: Optimizer (1 step!)
grads = clipped_grad_fn(params, batch)
params, state, metrics = step_fn(params, grads, state)  # ← Noise + update + accounting!
```

Let's see how this simplifies real training!

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

# Opaque imports
from opaque import (
  clipped_grad,
  add_gaussian_noise,
  RDPAccountant,
  calibrate_noise_multiplier,
)
# Optimizer imports
from opaque.optimizers import dp_sgd, dp_adamw, dp_adamw_ac

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print("Opaque imported successfully!")

## Part 1: Setup - Same as Tutorial 03

Let's use the same data and model from Tutorial 03 for easy comparison.

In [ ]:
# Generate synthetic binary classification data
def generate_data(n_samples=1000, n_features=20, seed=42):
  torch.manual_seed(seed)
  X = torch.randn(n_samples, n_features)
  y = (X[:, 0] + 0.5 * X[:, 1] > 0).float()
  return X, y


# Generate training data
X_train, y_train = generate_data(n_samples=10000)
X_test, y_test = generate_data(n_samples=300, seed=123)

# Initialize model parameters (logistic regression)
n_features = X_train.shape[1]
params_init = {
  'weight': torch.randn(n_features, 1) * 0.01,
  'bias': torch.zeros(1),
}


# Define loss function
def loss_fn(params, x, y):
  """Binary cross-entropy loss for logistic regression."""
  logits = x @ params['weight'] + params['bias']
  return F.binary_cross_entropy_with_logits(logits.squeeze(-1), y)


print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Model: Logistic Regression")
print(f"Parameters: {sum(p.numel() for p in params_init.values())}")

In [ ]:
# Training configuration (same as Tutorial 03)
target_epsilon = 3.0
target_delta = 0.5e-5
l2_clip_norm = 1.0
batch_size = 50
n_epochs = 20  # Shorter for demo
learning_rate = 0.1

# Privacy setup
sample_rate = batch_size / len(X_train)
steps_per_epoch = len(X_train) // batch_size
total_steps = n_epochs * steps_per_epoch

noise_multiplier = calibrate_noise_multiplier(
  target_epsilon=target_epsilon,
  target_delta=target_delta,
  sample_rate=sample_rate,
  num_steps=total_steps,
  accountant_type="rdp",
)

print(f"Training Configuration:")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {n_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Clip norm: {l2_clip_norm}")
print(f"  Noise multiplier: {noise_multiplier:.4f}")
print(f"\nPrivacy Target: (ε={target_epsilon}, δ={target_delta})")

## Part 2: Recap - Manual DP-SGD (Tutorial 03)

Let's quickly review the manual approach from Tutorial 03:

In [ ]:
print("Manual DP-SGD Training (Tutorial 03 approach):")
print("=" * 60)

# Copy parameters
params_manual = {k: v.clone() for k, v in params_init.items()}

# Setup
clipped_grad_fn = clipped_grad(
  loss_fn,
  argnums=0,
  batch_argnums=(1, 2),
  l2_clip_norm=l2_clip_norm,
)
accountant_manual = RDPAccountant()
noise_gen = torch.Generator().manual_seed(42)
stddev = noise_multiplier * l2_clip_norm

losses_manual = []
epsilons_manual = []

for epoch in range(n_epochs):
  perm = torch.randperm(len(X_train))
  X_shuffled = X_train[perm]
  y_shuffled = y_train[perm]

  epoch_loss = 0.0
  n_batches = 0

  for i in range(0, len(X_train), batch_size):
    X_batch = X_shuffled[i:i + batch_size]
    y_batch = y_shuffled[i:i + batch_size]

    # Manual DP-SGD (3 steps)
    # Step 1: Clipped gradients
    grads = clipped_grad_fn(params_manual, X_batch, y_batch)

    # Step 2: Add noise
    grads = add_gaussian_noise(grads, stddev=stddev, generator=noise_gen)

    # Step 3: Manual SGD update
    batch_size_actual = len(X_batch)
    for key in params_manual:
      params_manual[key] = params_manual[key] - (learning_rate / batch_size_actual) * grads[key]

    # Logging
    with torch.no_grad():
      epoch_loss += loss_fn(params_manual, X_batch, y_batch).item()
      n_batches += 1

  # Account for privacy
  accountant_manual.step_poisson(noise_multiplier, sample_rate, num_steps=steps_per_epoch)
  epsilon = accountant_manual.get_epsilon(target_delta)

  avg_loss = epoch_loss / n_batches
  losses_manual.append(avg_loss)
  epsilons_manual.append(epsilon)

  if (epoch + 1) % 5 == 0:
    print(f"Epoch {epoch + 1}/{n_epochs}: loss={avg_loss:.4f}, ε={epsilon:.2f}")

print(f"\n✓ Manual DP-SGD complete")
print(f"  Final loss: {losses_manual[-1]:.4f}")
print(f"  Privacy: ε={epsilons_manual[-1]:.2f}")

### What's Wrong with Manual?

Nothing! But it's **verbose** and **error-prone**:

- 😰 **Manual noise injection**: Easy to forget or misconfigure
- 😰 **Manual accounting**: Must call `accountant.step()` yourself
- 😰 **Manual updates**: Implementing SGD/Adam/AdamW from scratch
- 😰 **No momentum**: Standard SGD is slow for complex models

**Solution**: Use Opaque's DP optimizers! 🚀

---

## Part 3: DP-SGD Optimizer

Let's use `dp_sgd()` - it handles noise, updates, and accounting automatically!

In [ ]:
print("DP-SGD Optimizer:")
print("=" * 60)

# Copy parameters
params_dpsgd = {k: v.clone() for k, v in params_init.items()}

# Create clipped gradient function (same as before)
clipped_grad_fn = clipped_grad(
  loss_fn,
  argnums=0,
  batch_argnums=(1, 2),
  l2_clip_norm=l2_clip_norm,
)

# ✅ Create DP-SGD optimizer
init_fn, step_fn = dp_sgd(
  learning_rate=learning_rate,
  momentum=0.0,  # No momentum for now
  l2_clip_norm=l2_clip_norm,
  noise_multiplier=noise_multiplier,
  sample_rate=sample_rate,
  target_delta=target_delta,
)

# Initialize optimizer state
state = init_fn(params_dpsgd)

losses_dpsgd = []
epsilons_dpsgd = []

for epoch in range(n_epochs):
  perm = torch.randperm(len(X_train))
  X_shuffled = X_train[perm]
  y_shuffled = y_train[perm]

  epoch_loss = 0.0
  n_batches = 0

  for i in range(0, len(X_train), batch_size):
    X_batch = X_shuffled[i:i + batch_size]
    y_batch = y_shuffled[i:i + batch_size]

    # Compute clipped gradients
    grads = clipped_grad_fn(params_dpsgd, X_batch, y_batch)

    # ✅ ONE LINE: Noise + update + accounting!
    params_dpsgd, state, metrics = step_fn(params_dpsgd, grads, state)

    # Logging
    with torch.no_grad():
      epoch_loss += loss_fn(params_dpsgd, X_batch, y_batch).item()
      n_batches += 1

  avg_loss = epoch_loss / n_batches
  losses_dpsgd.append(avg_loss)
  epsilons_dpsgd.append(metrics['epsilon'])

  if (epoch + 1) % 5 == 0:
    print(f"Epoch {epoch + 1}/{n_epochs}: loss={avg_loss:.4f}, ε={metrics['epsilon']:.2f}")

print(f"\n✓ DP-SGD optimizer complete")
print(f"  Final loss: {losses_dpsgd[-1]:.4f}")
print(f"  Privacy: ε={epsilons_dpsgd[-1]:.2f}")

### 🎯 Key Benefits

Compare the optimizer approach to manual:

```python
# Manual (3 lines per step)
grads = clipped_grad_fn(params, batch)
grads = add_gaussian_noise(grads, stddev, generator)  # ← Easy to forget!
params = manual_update(params, grads, lr)             # ← Implement yourself
accountant.step_poisson(...)                          # ← Easy to forget!

# Optimizer (1 line!)
grads = clipped_grad_fn(params, batch)
params, state, metrics = step_fn(params, grads, state)  # ← All automatic!
```

**Automatic**:
- ✅ Noise injection (correct stddev)
- ✅ Parameter updates (SGD logic)
- ✅ Privacy accounting (tracks ε)
- ✅ Metrics (epsilon, step count)

**Result**: Identical privacy and performance, cleaner code!

---

In [ ]:
# Verify they're equivalent
print("Verification: Manual vs Optimizer")
print("=" * 60)
print(f"Manual final loss:    {losses_manual[-1]:.6f}")
print(f"Optimizer final loss: {losses_dpsgd[-1]:.6f}")
print(f"Difference:           {abs(losses_manual[-1] - losses_dpsgd[-1]):.6f}")
print(f"\nManual final ε:       {epsilons_manual[-1]:.4f}")
print(f"Optimizer final ε:    {epsilons_dpsgd[-1]:.4f}")
print(f"Difference:           {abs(epsilons_manual[-1] - epsilons_dpsgd[-1]):.4f}")
print(f"\n✓ Nearly identical results (small differences due to randomness)")

## Part 4: DP-AdamW - Adaptive Learning Rates

SGD is great, but **Adam** is the standard for neural networks. Let's use `dp_adamw()`!

### Why AdamW?

**Standard SGD**:
```
θ ← θ - η·g
```

**Adam** (adaptive moments):
```
m ← β₁·m + (1-β₁)·g        # First moment (momentum)
v ← β₂·v + (1-β₂)·g²       # Second moment (adaptive LR)
θ ← θ - η·m / (√v + ε)
```

**AdamW** (Adam + weight decay):
```
θ ← θ - η·m / (√v + ε) - η·λ·θ   # Explicit regularization
```

**Benefits**:
- ⚡ **Faster convergence**: Adapts learning rate per parameter
- 🎯 **Better for LLMs**: Weight decay as explicit regularization
- 📈 **More stable**: Momentum smooths noisy gradients

Let's try it!

In [ ]:
print("DP-AdamW Optimizer:")
print("=" * 60)

# Copy parameters
params_adamw = {k: v.clone() for k, v in params_init.items()}

# Same clipped gradient function
clipped_grad_fn = clipped_grad(
  loss_fn,
  argnums=0,
  batch_argnums=(1, 2),
  l2_clip_norm=l2_clip_norm,
)

# ✅ Create DP-AdamW optimizer
init_fn_adamw, step_fn_adamw = dp_adamw(
  learning_rate=learning_rate,
  betas=(0.9, 0.999),  # Adam momentum parameters
  weight_decay=0.01,  # Regularization
  l2_clip_norm=l2_clip_norm,
  noise_multiplier=noise_multiplier,
  sample_rate=sample_rate,
  target_delta=target_delta,
)

# Initialize optimizer state
state_adamw = init_fn_adamw(params_adamw)

losses_adamw = []
epsilons_adamw = []

for epoch in range(n_epochs):
  perm = torch.randperm(len(X_train))
  X_shuffled = X_train[perm]
  y_shuffled = y_train[perm]

  epoch_loss = 0.0
  n_batches = 0

  for i in range(0, len(X_train), batch_size):
    X_batch = X_shuffled[i:i + batch_size]
    y_batch = y_shuffled[i:i + batch_size]

    # Compute clipped gradients
    grads = clipped_grad_fn(params_adamw, X_batch, y_batch)

    # ✅ DP-AdamW step (same API as dp_sgd!)
    params_adamw, state_adamw, metrics = step_fn_adamw(params_adamw, grads, state_adamw)

    # Logging
    with torch.no_grad():
      epoch_loss += loss_fn(params_adamw, X_batch, y_batch).item()
      n_batches += 1

  avg_loss = epoch_loss / n_batches
  losses_adamw.append(avg_loss)
  epsilons_adamw.append(metrics['epsilon'])

  if (epoch + 1) % 5 == 0:
    print(f"Epoch {epoch + 1}/{n_epochs}: loss={avg_loss:.4f}, ε={metrics['epsilon']:.2f}")

print(f"\n✓ DP-AdamW complete")
print(f"  Final loss: {losses_adamw[-1]:.4f}")
print(f"  Privacy: ε={epsilons_adamw[-1]:.2f}")

In [ ]:
# Compare all three
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training loss
ax1.plot(losses_manual, label='Manual DP-SGD', marker='o', linewidth=2)
ax1.plot(losses_dpsgd, label='DP-SGD Optimizer', marker='s', linewidth=2, linestyle='--')
ax1.plot(losses_adamw, label='DP-AdamW', marker='^', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Loss Comparison', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Plot 2: Privacy consumption
ax2.plot(epsilons_manual, label='Manual DP-SGD', marker='o', linewidth=2)
ax2.plot(epsilons_dpsgd, label='DP-SGD Optimizer', marker='s', linewidth=2, linestyle='--')
ax2.plot(epsilons_adamw, label='DP-AdamW', marker='^', linewidth=2)
ax2.axhline(y=target_epsilon, color='red', linestyle=':', linewidth=2, label=f'Target ε={target_epsilon}')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Epsilon (ε)', fontsize=12)
ax2.set_title('Privacy Budget', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("  1. Manual and DP-SGD optimizer are nearly identical (as expected)")
print("  2. DP-AdamW may converge faster with adaptive learning rates")
print("  3. All have same privacy cost (same noise multiplier)")

### When to Use Each Optimizer?

| Optimizer | Best For | Why |
|-----------|----------|-----|
| **DP-SGD** | Simple models, convex problems | Fast, stable, easy to tune |
| **DP-AdamW** | Neural networks, LLMs, LoRA | Adaptive LR, weight decay, faster convergence |

**Rule of thumb**: Use DP-AdamW for everything except the simplest models!

---

## Part 5 (Extra): DP-AdamW-AC - Adaptive Clipping

So far, we've used **fixed clipping**: clip norm C stays constant during training.

But what if:
- 🤔 Early training: Large gradients (need larger C)
- 🤔 Late training: Small gradients (smaller C preserves more signal)

**Solution**: **Adaptive clipping** adjusts C automatically!

### DP-AdamW-AC Algorithm

From [Zuo et al., "DP-Adam-AC", October 2024](https://arxiv.org/abs/2510.05288):

1. **Track gradient norms**: Keep buffer of recent per-example gradient norms
2. **Adapt clip norm**: `C ← Percentile_q(buffer)` where `q = 1 - ρ*`
3. **Adjust learning rate**: Scale LR based on observed clip rate ρ
4. **EMA parameters**: Smooth parameter trajectory for better generalization

**Result**: 1-3% better accuracy with same privacy!

Let's try it!

In [ ]:
print("DP-AdamW-AC (Adaptive Clipping):")
print("=" * 60)

# Copy parameters
params_ac = {k: v.clone() for k, v in params_init.items()}

# ⚠️ Important: Need gradient norms for adaptive clipping!
# Use return_grad_norms=True to get pre-clip norms
initial_clip_norm = 3.0  # Start higher than optimal

clipped_grad_fn_ac = clipped_grad(
  loss_fn,
  argnums=0,
  batch_argnums=(1, 2),
  l2_clip_norm=initial_clip_norm,  # Will be updated adaptively!
  return_grad_norms=True,  # ← Need this for adaptive clipping
)

# ✅ Create DP-AdamW-AC optimizer
init_fn_ac, step_fn_ac = dp_adamw_ac(
  learning_rate=learning_rate,
  weight_decay=0.01,  # AdamW regularization
  initial_clip_norm=initial_clip_norm,
  noise_multiplier=noise_multiplier,
  sample_rate=sample_rate,
  target_delta=target_delta,
  target_clip_rate=0.20,  # Aim for 20% clipping
  history_size=1000,  # Buffer size
  clip_norm_min=0.1,
  clip_norm_max=10.0,
)

# Initialize optimizer state
state_ac = init_fn_ac(params_ac)

losses_ac = []
epsilons_ac = []
clip_norms_ac = []
clip_rates_ac = []
lr_mults_ac = []

print(f"Initial clip norm: {initial_clip_norm}")
print(f"Training...\n")

for epoch in range(n_epochs):
  perm = torch.randperm(len(X_train))
  X_shuffled = X_train[perm]
  y_shuffled = y_train[perm]

  epoch_loss = 0.0
  n_batches = 0

  for i in range(0, len(X_train), batch_size):
    X_batch = X_shuffled[i:i + batch_size]
    y_batch = y_shuffled[i:i + batch_size]

    # Update clipping threshold to current adaptive value
    # Note: This pattern updates the frozen function's clip norm for each iteration
    clipped_grad_fn_ac.keywords['l2_clip_norm'] = state_ac.current_clip_norm

    # Compute clipped gradients + norms
    grads, aux = clipped_grad_fn_ac(params_ac, X_batch, y_batch)
    grad_norms = aux.grad_norms  # Pre-clip norms

    # ✅ DP-AdamW-AC step (batch_sizes optional for standard training)
    params_ac, state_ac, metrics = step_fn_ac(
      params_ac, grads, grad_norms, state_ac  # batch_sizes defaults to ones
    )

    # Logging
    with torch.no_grad():
      epoch_loss += loss_fn(params_ac, X_batch, y_batch).item()
      n_batches += 1

  avg_loss = epoch_loss / n_batches
  losses_ac.append(avg_loss)
  epsilons_ac.append(metrics['epsilon'])
  clip_norms_ac.append(metrics['clip_norm'])
  clip_rates_ac.append(metrics['clip_rate'])
  lr_mults_ac.append(metrics['lr_multiplier'])

  if (epoch + 1) % 5 == 0:
    print(f"Epoch {epoch + 1}/{n_epochs}:")
    print(f"  loss={avg_loss:.4f}, ε={metrics['epsilon']:.2f}")
    print(f"  C={metrics['clip_norm']:.2f}, ρ={metrics['clip_rate']:.1%}, γ={metrics['lr_multiplier']:.2f}")

print(f"\n✓ DP-AdamW-AC complete")
print(f"  Final loss: {losses_ac[-1]:.4f}")
print(f"  Privacy: ε={epsilons_ac[-1]:.2f}")
print(f"  Final clip norm: {clip_norms_ac[-1]:.2f} (started at {initial_clip_norm})")

In [ ]:
# Visualize adaptive behavior
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Loss comparison
ax1.plot(losses_dpsgd, label='DP-SGD (fixed C)', marker='o', linewidth=2)
ax1.plot(losses_adamw, label='DP-AdamW (fixed C)', marker='s', linewidth=2)
ax1.plot(losses_ac, label='DP-AdamW-AC (adaptive C)', marker='^', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Loss Comparison', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Plot 2: Adaptive clip norm
ax2.plot(clip_norms_ac, marker='o', linewidth=2, color='red')
ax2.axhline(y=l2_clip_norm, color='blue', linestyle='--', linewidth=2, label=f'Fixed C={l2_clip_norm}')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Clip Norm (C)', fontsize=12)
ax2.set_title('Adaptive Clipping Threshold', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# Plot 3: Clip rate
ax3.plot(clip_rates_ac, marker='o', linewidth=2, color='green')
ax3.axhline(y=0.20, color='black', linestyle='--', linewidth=2, label='Target ρ=20%')
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('Clip Rate (ρ)', fontsize=12)
ax3.set_title('Fraction of Clipped Gradients', fontsize=14)
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

# Plot 4: Learning rate multiplier
ax4.plot(lr_mults_ac, marker='o', linewidth=2, color='orange')
ax4.axhline(y=1.0, color='black', linestyle='--', linewidth=2, label='Neutral γ=1.0')
ax4.set_xlabel('Epoch', fontsize=12)
ax4.set_ylabel('LR Multiplier (γ)', fontsize=12)
ax4.set_title('Dynamic Learning Rate Scaling', fontsize=14)
ax4.legend(fontsize=10)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print(f"  1. Clip norm C adapted from {initial_clip_norm} → {clip_norms_ac[-1]:.2f}")
print(f"  2. Clip rate stabilized around target 20%")
print(f"  3. Learning rate adjusted automatically based on clipping")
print(f"  4. Adaptive clipping can improve convergence!")

### 🎯 When to Use Adaptive Clipping?

**Use DP-AdamW-AC when**:
- ✅ Training complex models (neural networks, transformers)
- ✅ Gradient norms vary significantly during training
- ✅ Want to squeeze out extra 1-3% accuracy
- ✅ Have compute budget for gradient norm tracking

**Stick with fixed clipping when**:
- ⚠️ Simple models (logistic regression, linear models)
- ⚠️ You know the right clip norm from prior experience
- ⚠️ Want simplest possible code

**Trade-offs**:
- **Pros**: Better privacy-utility, automatic tuning, state-of-the-art
- **Cons**: More complex, requires gradient norms, more hyperparameters

---

## Summary

### What We Learned

You now have **three** production-ready DP optimizers:

| Optimizer | Complexity | Best For | Key Feature |
|-----------|-----------|----------|-------------|
| **`dp_sgd()`** | Simple | Basic models | Momentum optional |
| **`dp_adamw()`** | Medium | Neural nets, LLMs | Adaptive LR + weight decay |
| **`dp_adamw_ac()`** | Advanced | LLMs + state-of-the-art | Adaptive clipping + weight decay |

### The Complete Pattern

```python
# 1. Create clipped gradient function
clipped_grad_fn = clipped_grad(
    loss_fn, argnums=0, batch_argnums=(1, 2), l2_clip_norm=1.0
)

# 2. Create optimizer
init_fn, step_fn = dp_adamw(
    learning_rate=1e-3,
    weight_decay=0.01,
    l2_clip_norm=1.0,
    noise_multiplier=1.1,
    sample_rate=0.01,
    target_delta=1e-5,
)

# 3. Initialize
state = init_fn(params)

# 4. Training loop
for batch in dataloader:
    grads = clipped_grad_fn(params, batch)
    params, state, metrics = step_fn(params, grads, state)
    print(f"ε={metrics['epsilon']:.2f}")
```

That's it! **One line per step** instead of manual noise + updates + accounting.

---

### Key Takeaways

1. **DP optimizers are drop-in**: Same API, automatic noise + accounting
2. **Start with dp_adamw**: Best for most use cases
3. **Use dp_adamw_ac for max performance**: State-of-the-art adaptive clipping + weight decay
4. **All have same privacy cost**: Same noise_multiplier → same ε

---

### What's Next?

**Tutorial 05: DP-SGD for LoRA Fine-Tuning with HuggingFace**
- Apply DP-AdamW to real LLMs (GPT-2, LLaMA)
- LoRA + differential privacy
- Production-ready patterns

---

🎉 **Congratulations!** You can now train with production-ready DP optimizers!

**Resources**:
- [DP-SGD Paper](https://arxiv.org/abs/1607.00133) (Abadi et al., 2016)
- [DP-Adam-AC Paper](https://arxiv.org/abs/2510.05288) (Zuo et al., 2024)
- [TorchOpt Documentation](https://torchopt.readthedocs.io/)
- [Opaque API Reference](../../api/optimizers.md)